In [8]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V1 (Analytic - Optimized)
========================================================================
Optimizations applied:
- Removed visualize_saliency_map
- Vectorized histogram/profile computations
- Parallelized m/z signature extraction
- Cached NearestNeighbors fits
- Batch processing with pre-computed alignments
"""

import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from typing import Dict, Optional
import pandas as pd
import os
import warnings
from dataclasses import dataclass
import pickle
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')

# =============================================================================
# DATA CONFIGURATION
# =============================================================================

RNA_PIXEL_SIZE = 55  # μm (Visium)
MSI_PIXEL_SIZE = 60  # μm
TOP_K_MATCHES = 26
KNN_INTERP_K=6

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]


RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_all/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]

RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

#AAD_TARGET_GENES = ['Thy1', 'App', 'Apoe', 'Mapt']

"""AAD_TARGET_GENES = ['Eno1', 'Mapt', 'Thy1', 'Pmch', 'Atp1a3', 'Rac1', 'Rsrp1', 'Snhg11', 'Tubb4a',
    'Rasgrf1', 'Hsp90ab1', 'Elavl3', 'App', 'Syp', 'AC149090.1',
    'Aplp1', 'Apoe', 'Meg3', 'Gnas', 'Pkm']"""

AAD_TARGET_GENES = ['App', 'Psen1', 'Psen2', 'Mapt', 'Apoe', 'Trem2', 'Tyrobp',
 'C1qa', 'C1qb', 'C1qc', 'Csf1r', 'Aif1', 'Cst7',
 'Sele', 'Ccr6', 'Aqp4', 'Plcg2', 'Abca7', 'Bace1']


'''['App', 'Mapt', 'Syp', 'Apoe','Cst3', 'Eno1', 'Thy1', 'Pmch', 
                    'Atp1a3', 'Rac1', 'Rsrp1', 'Snhg11', 'Tubb4a',
                    'Rasgrf1', 'Hsp90ab1', 'Elavl3', 'AC149090.1',
                    'Aplp1', 'Meg3', 'Gnas', 'Pkm']'''


def rotate_180(coords: np.ndarray) -> np.ndarray:
    center = coords.mean(axis=0)
    return 2 * center - coords


def align_rna_to_msi(rna_coords: np.ndarray, msi_coords: np.ndarray) -> np.ndarray:
    rotated = rotate_180(rna_coords)
    rna_min, rna_max = rotated.min(axis=0), rotated.max(axis=0)
    msi_min, msi_max = msi_coords.min(axis=0), msi_coords.max(axis=0)
    rna_range = rna_max - rna_min
    msi_range = msi_max - msi_min
    scale = msi_range / (rna_range + 1e-8)
    return (rotated - rna_min) * scale + msi_min


def compute_value_histogram(values: np.ndarray, n_bins: int = 50) -> np.ndarray:
    hist, _ = np.histogram(values, bins=n_bins, density=True)
    return hist / (hist.sum() + 1e-8)


def compute_spatial_histogram(coords: np.ndarray, values: np.ndarray, n_bins: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_bins).astype(int), 0, n_bins - 1)
    y_bins = np.clip((norm[:, 1] * n_bins).astype(int), 0, n_bins - 1)
    flat_idx = y_bins * n_bins + x_bins
    hist = np.bincount(flat_idx, weights=values, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    counts = np.bincount(flat_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    hist_min, hist_max = hist.min(), hist.max()
    if hist_max > hist_min:
        hist = (hist - hist_min) / (hist_max - hist_min)
    return hist


def compute_radial_profile(coords: np.ndarray, values: np.ndarray, n_rings: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    centroid = norm.mean(axis=0)
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max() + 1e-8
    ring_idx = np.clip((distances / max_dist * n_rings).astype(int), 0, n_rings - 1)
    profile = np.bincount(ring_idx, weights=values, minlength=n_rings)
    counts = np.bincount(ring_idx, minlength=n_rings)
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile, dtype=float))
    prof_min, prof_max = profile.min(), profile.max()
    if prof_max > prof_min:
        profile = (profile - prof_min) / (prof_max - prof_min)
    return profile


def compute_quadrant_stats(coords: np.ndarray, values: np.ndarray, n_div: int = 3) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_div).astype(int), 0, n_div - 1)
    y_bins = np.clip((norm[:, 1] * n_div).astype(int), 0, n_div - 1)
    flat_idx = y_bins * n_div + x_bins
    stats = np.zeros(n_div * n_div * 2)
    for q in range(n_div * n_div):
        mask = flat_idx == q
        if mask.sum() > 0:
            q_vals = values[mask]
            stats[q * 2] = np.mean(q_vals)
            stats[q * 2 + 1] = np.std(q_vals)
    stats_min, stats_max = stats.min(), stats.max()
    if stats_max > stats_min:
        stats = (stats - stats_min) / (stats_max - stats_min)
    return stats


def compute_morans_i_vectorized(coords: np.ndarray, values: np.ndarray, indices: np.ndarray) -> float:
    n = len(values)
    mean_val = values.mean()
    deviations = values - mean_val
    denom = np.sum(deviations ** 2)
    if denom == 0:
        return 0.0
    neighbor_deviations = deviations[indices[:, 1:]]
    numer = np.sum(deviations[:, np.newaxis] * neighbor_deviations)
    w_sum = indices.shape[0] * (indices.shape[1] - 1)
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0.0


def precompute_grid_idw_weights(coords, grid_points_flat, k=KNN_INTERP_K):
    nn = NearestNeighbors(n_neighbors=k, algorithm='ball_tree')
    nn.fit(coords)
    distances, indices = nn.kneighbors(grid_points_flat)
    distances = np.maximum(distances, 1e-10)
    weights = 1.0 / (distances ** 2)
    weight_sums = weights.sum(axis=1, keepdims=True)
    weights = weights / weight_sums
    return indices, weights

def idw_interpolate(values, nn_indices, nn_weights):
    neighbor_vals = values[nn_indices]
    return np.sum(neighbor_vals * nn_weights, axis=1)


@dataclass
class SpatialSignature:
    sample_id: str
    feature_name: str
    feature_type: str
    node_importance: np.ndarray
    value_histogram: np.ndarray = None
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    quadrant_stats: np.ndarray = None
    morans_i: float = 0.0
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None
    aligned_coordinates: Optional[np.ndarray] = None


def compute_coordinate_based_similarity(gene_sig, mz_sig, grid_res=50):
    gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
    mz_coords = mz_sig.coordinates

    # Shared grid spanning both coordinate ranges (keeps original logic)
    x_min = min(gene_coords[:, 0].min(), mz_coords[:, 0].min())
    x_max = max(gene_coords[:, 0].max(), mz_coords[:, 0].max())
    y_min = min(gene_coords[:, 1].min(), mz_coords[:, 1].min())
    y_max = max(gene_coords[:, 1].max(), mz_coords[:, 1].max())

    gx, gy = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    grid_flat = np.column_stack([gx.ravel(), gy.ravel()])

    # IDW interpolation for gene
    gene_idw_idx, gene_idw_wts = precompute_grid_idw_weights(gene_coords, grid_flat, k=KNN_INTERP_K)
    gene_grid = idw_interpolate(gene_sig.raw_values, gene_idw_idx, gene_idw_wts)
    gene_imp_grid = idw_interpolate(gene_sig.node_importance, gene_idw_idx, gene_idw_wts)

    # IDW interpolation for m/z
    mz_idw_idx, mz_idw_wts = precompute_grid_idw_weights(mz_coords, grid_flat, k=KNN_INTERP_K)
    mz_grid = idw_interpolate(mz_sig.raw_values, mz_idw_idx, mz_idw_wts)
    mz_imp_grid = idw_interpolate(mz_sig.node_importance, mz_idw_idx, mz_idw_wts)

    # Value correlation (no NaN masking needed with IDW)
    value_corr = 0.0
    if len(gene_grid) > 10:
        a, b = gene_grid - gene_grid.mean(), mz_grid - mz_grid.mean()
        ss_a, ss_b = np.dot(a, a), np.dot(b, b)
        if ss_a > 0 and ss_b > 0:
            value_corr = np.dot(a, b) / np.sqrt(ss_a * ss_b)

    # Importance metrics
    gi = gene_imp_grid.copy()
    mi = mz_imp_grid.copy()
    gi_max, mi_max = gi.max(), mi.max()
    if gi_max > 0:
        gi = gi / gi_max
    if mi_max > 0:
        mi = mi / mi_max

    importance_iou = np.minimum(gi, mi).sum() / (np.maximum(gi, mi).sum() + 1e-8)

    imp_corr = 0.0
    a, b = gi - gi.mean(), mi - mi.mean()
    ss_a, ss_b = np.dot(a, a), np.dot(b, b)
    if ss_a > 0 and ss_b > 0:
        imp_corr = np.dot(a, b) / np.sqrt(ss_a * ss_b)

    return {
        'value_correlation': value_corr,
        'importance_iou': importance_iou,
        'importance_correlation': imp_corr
    }

def compute_descriptor_similarity(gene_sig: SpatialSignature, mz_sig: SpatialSignature) -> dict:
    def safe_pearsonr(a, b):
        if a.std() > 0 and b.std() > 0:
            r, _ = pearsonr(a, b)
            return r if not np.isnan(r) else 0
        return 0
    val_hist_corr = safe_pearsonr(gene_sig.value_histogram, mz_sig.value_histogram)
    spatial_hist_corr = safe_pearsonr(gene_sig.spatial_histogram.flatten(), mz_sig.spatial_histogram.flatten())
    radial_corr = safe_pearsonr(gene_sig.radial_profile, mz_sig.radial_profile)
    quad_corr = safe_pearsonr(gene_sig.quadrant_stats, mz_sig.quadrant_stats)
    morans_sim = 1.0 - abs(gene_sig.morans_i - mz_sig.morans_i)
    return {'value_hist_corr': val_hist_corr, 'spatial_hist_corr': spatial_hist_corr, 
            'radial_corr': radial_corr, 'quadrant_corr': quad_corr, 'morans_similarity': morans_sim}


def compute_combined_score(coord_sim: dict, desc_sim: dict) -> float:
    coord_score = (0.0832 * max(coord_sim['value_correlation'], 0) +
                   0.1102 * coord_sim['importance_iou'] +
                   0.1356 * max(coord_sim['importance_correlation'], 0))
    desc_score = (0.1000 * max(desc_sim['spatial_hist_corr'], 0) +
                  0.1814 * max(desc_sim['radial_corr'], 0) +
                  0.1133 * max(desc_sim['quadrant_corr'], 0) +
                  0.0914 * max(desc_sim['morans_similarity'], 0) +
                  0.1850 * max(desc_sim['value_hist_corr'], 0))
    return coord_score + desc_score


class AnalyticPatternMatcher:
    def __init__(self, output_dir: str = './gene_to_mz_results_v1_analytic', n_jobs: int = -1):
        self.output_dir = output_dir
        self.n_jobs = n_jobs
        for subdir in ['gene_visualizations', 'match_visualizations', 'descriptors']:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        self.rna_data = {}
        self.msi_data = {}
        self._nn_cache = {}

    def load_all_data(self):
        print("Loading RNA-seq data...")
        print(f"  Pixel size: {RNA_PIXEL_SIZE} μm (hexagonal)")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        print(f"\nLoading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")

    def _get_nn_indices(self, coords: np.ndarray, k: int, cache_key: str) -> np.ndarray:
        full_key = f"{cache_key}_{k}"
        if full_key not in self._nn_cache:
            coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
            norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
            k_actual = min(k + 1, len(coords))
            nn = NearestNeighbors(n_neighbors=k_actual)
            nn.fit(norm)
            _, indices = nn.kneighbors(norm)
            self._nn_cache[full_key] = indices
        return self._nn_cache[full_key]

    def compute_bio_importance(self, coords: np.ndarray, values: np.ndarray, k: int, nn_indices: np.ndarray) -> np.ndarray:
        neighbor_vals = values[nn_indices[:, 1:]]
        local_var = np.var(neighbor_vals, axis=1)
        lv_min, lv_max = local_var.min(), local_var.max()
        if lv_max > lv_min:
            local_var = (local_var - lv_min) / (lv_max - lv_min)
        else:
            local_var = np.zeros_like(local_var)
        v_min, v_max = values.min(), values.max()
        if v_max > v_min:
            val_norm = (values - v_min) / (v_max - v_min)
        else:
            val_norm = np.zeros_like(values)
        return 0.5 * local_var + 0.5 * val_norm

    def extract_signature(self, coords: np.ndarray, values: np.ndarray, sample_id: str,
                          feature_name: str, feature_type: str, n_neighbors: int,
                          nn_indices: np.ndarray, aligned_coords: Optional[np.ndarray] = None) -> SpatialSignature:
        bio_imp = self.compute_bio_importance(coords, values, n_neighbors, nn_indices)
        return SpatialSignature(
            sample_id=sample_id, feature_name=feature_name, feature_type=feature_type,
            node_importance=bio_imp, value_histogram=compute_value_histogram(values),
            spatial_histogram=compute_spatial_histogram(coords, values),
            radial_profile=compute_radial_profile(coords, values),
            quadrant_stats=compute_quadrant_stats(coords, values),
            morans_i=compute_morans_i_vectorized(coords, values, nn_indices),
            coordinates=coords, raw_values=values, aligned_coordinates=aligned_coords)

    def visualize_signature(self, sig: SpatialSignature, save_path: str):
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        im1 = axes[0, 0].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                 c=sig.raw_values, cmap='viridis', s=10)
        axes[0, 0].set_title(f'{sig.feature_name} - Original', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0], label='Expression')
        if sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(sig.aligned_coordinates[:, 0], sig.aligned_coordinates[:, 1],
                                     c=sig.raw_values, cmap='viridis', s=10)
            axes[0, 1].set_title('Aligned (180° rotated)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='Expression')
        else:
            log_vals = np.log1p(sig.raw_values)
            im2 = axes[0, 1].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                     c=log_vals, cmap='viridis', s=10)
            axes[0, 1].set_title('Log-transformed', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='log(1+x)')
        axes[0, 1].set_xlabel('X (μm)')
        axes[0, 1].set_ylabel('Y (μm)')
        im3 = axes[0, 2].imshow(sig.spatial_histogram, cmap='viridis', origin='lower', aspect='auto')
        axes[0, 2].set_title('Spatial Histogram (10x10)', fontweight='bold')
        axes[0, 2].set_xlabel('X bin')
        axes[0, 2].set_ylabel('Y bin')
        plt.colorbar(im3, ax=axes[0, 2], label='Mean Expression')
        axes[0, 3].plot(sig.radial_profile, 'b-', linewidth=2, marker='o', markersize=4)
        axes[0, 3].fill_between(range(len(sig.radial_profile)), sig.radial_profile, alpha=0.3)
        axes[0, 3].set_title('Radial Profile (from centroid)', fontweight='bold')
        axes[0, 3].set_xlabel('Ring (center → edge)')
        axes[0, 3].set_ylabel('Normalized Expression')
        axes[0, 3].grid(True, alpha=0.3)
        axes[1, 0].bar(range(len(sig.value_histogram)), sig.value_histogram,
                       color='steelblue', edgecolor='darkblue', alpha=0.7)
        axes[1, 0].set_title('Expression Distribution', fontweight='bold')
        axes[1, 0].set_xlabel('Bin')
        axes[1, 0].set_ylabel('Density')
        n_div = 3
        quad_means = sig.quadrant_stats[::2].reshape(n_div, n_div)
        im4 = axes[1, 1].imshow(quad_means, cmap='YlOrRd', origin='lower', aspect='auto')
        axes[1, 1].set_title('Quadrant Mean Expression', fontweight='bold')
        axes[1, 1].set_xlabel('X quadrant')
        axes[1, 1].set_ylabel('Y quadrant')
        for i in range(n_div):
            for j in range(n_div):
                axes[1, 1].text(j, i, f'{quad_means[i, j]:.2f}', ha='center', va='center', fontsize=9)
        plt.colorbar(im4, ax=axes[1, 1])
        coords = sig.aligned_coordinates if sig.aligned_coordinates is not None else sig.coordinates
        percentiles = [25, 50, 75, 90]
        colors = ['lightblue', 'steelblue', 'orange', 'red']
        axes[1, 2].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=3, alpha=0.3)
        for pct, color in zip(percentiles, colors):
            thresh = np.percentile(sig.raw_values, pct)
            mask = sig.raw_values >= thresh
            axes[1, 2].scatter(coords[mask, 0], coords[mask, 1], c=color, s=5, alpha=0.5, label=f'≥{pct}th pct')
        axes[1, 2].set_title('Expression Percentiles', fontweight='bold')
        axes[1, 2].set_xlabel('X (μm)')
        axes[1, 2].set_ylabel('Y (μm)')
        axes[1, 2].legend(loc='upper right', fontsize=8)
        stats_text = f"""
EXPRESSION STATISTICS
═══════════════════════════════

Feature: {sig.feature_name}
Sample:  {sig.sample_id}
Type:    {sig.feature_type.upper()}

Expression Values:
  Mean:   {sig.raw_values.mean():.4f}
  Std:    {sig.raw_values.std():.4f}
  Min:    {sig.raw_values.min():.4f}
  Max:    {sig.raw_values.max():.4f}
  Median: {np.median(sig.raw_values):.4f}

Spatial Metrics:
  Moran's I:     {sig.morans_i:.4f}
  Total Spots:   {len(sig.raw_values)}
  Non-zero:      {(sig.raw_values > 0).sum()}
        """
        axes[1, 3].text(0.05, 0.95, stats_text, transform=axes[1, 3].transAxes,
                        fontsize=10, verticalalignment='top', family='monospace',
                        bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))
        axes[1, 3].axis('off')
        plt.suptitle(f'EXPRESSION PATTERN: {sig.feature_name} ({sig.sample_id}) | Moran\'s I: {sig.morans_i:.3f}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()

    def visualize_match(self, gene_sig, mz_sig, coord_sim, desc_sim, save_path):
        combined = compute_combined_score(coord_sim, desc_sim)
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        im1 = axes[0, 0].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                 c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        if gene_sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                                     c=gene_sig.raw_values, cmap='viridis', s=15)
            axes[0, 1].set_title('Gene (180° Aligned)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1])
        im3 = axes[0, 2].imshow(gene_sig.spatial_histogram, cmap='viridis', origin='lower')
        axes[0, 2].set_title('Gene Spatial Hist', fontweight='bold')
        plt.colorbar(im3, ax=axes[0, 2])
        axes[0, 3].plot(gene_sig.radial_profile, 'b-', linewidth=2, label='Gene')
        axes[0, 3].plot(mz_sig.radial_profile, 'r--', linewidth=2, label='m/z')
        axes[0, 3].set_title(f'Radial (r={desc_sim["radial_corr"]:.3f})', fontweight='bold')
        axes[0, 3].legend()
        im4 = axes[1, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                 c=mz_sig.raw_values, cmap='viridis', s=5)
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 0])
        im5 = axes[1, 1].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                 c=mz_sig.node_importance, cmap='hot', s=5)
        axes[1, 1].set_title('m/z Importance', fontweight='bold')
        plt.colorbar(im5, ax=axes[1, 1])
        im6 = axes[1, 2].imshow(mz_sig.spatial_histogram, cmap='viridis', origin='lower')
        axes[1, 2].set_title('m/z Spatial Hist', fontweight='bold')
        plt.colorbar(im6, ax=axes[1, 2])
        diff = gene_sig.spatial_histogram - mz_sig.spatial_histogram
        im7 = axes[1, 3].imshow(diff, cmap='RdBu_r', origin='lower')
        axes[1, 3].set_title(f'Spatial Diff (r={desc_sim["spatial_hist_corr"]:.3f})', fontweight='bold')
        plt.colorbar(im7, ax=axes[1, 3])
        if gene_sig.aligned_coordinates is not None:
            axes[2, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                               c='blue', s=3, alpha=0.3, label='m/z')
            axes[2, 0].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                               c='red', s=10, alpha=0.5, label='Gene')
            axes[2, 0].set_title('Overlay (Gene aligned)', fontweight='bold')
            axes[2, 0].legend()
        gene_thresh = np.percentile(gene_sig.node_importance, 70)
        mz_thresh = np.percentile(mz_sig.node_importance, 70)
        gene_mask = gene_sig.node_importance >= gene_thresh
        mz_mask = mz_sig.node_importance >= mz_thresh
        if gene_sig.aligned_coordinates is not None:
            axes[2, 1].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                               c='blue', s=8, alpha=0.5, label='m/z top 30%')
            axes[2, 1].scatter(gene_sig.aligned_coordinates[gene_mask, 0],
                               gene_sig.aligned_coordinates[gene_mask, 1],
                               c='red', s=15, alpha=0.7, label='Gene top 30%')
            axes[2, 1].set_title('Top 30% Overlay', fontweight='bold')
            axes[2, 1].legend()
        axes[2, 2].bar(range(len(gene_sig.value_histogram)), gene_sig.value_histogram, alpha=0.5, label='Gene')
        axes[2, 2].bar(range(len(mz_sig.value_histogram)), mz_sig.value_histogram, alpha=0.5, label='m/z')
        axes[2, 2].set_title(f'Value Hist (r={desc_sim["value_hist_corr"]:.3f})', fontweight='bold')
        axes[2, 2].legend()
        metrics_text = f"""
COMBINED SCORE: {combined:.4f}
═══════════════════════════════════

COORDINATE-BASED:
  Value correlation:    {coord_sim['value_correlation']:.4f}
  Importance IoU:       {coord_sim['importance_iou']:.4f}
  Importance corr:      {coord_sim['importance_correlation']:.4f}

DESCRIPTOR-BASED:
  Spatial hist corr:    {desc_sim['spatial_hist_corr']:.4f}
  Radial corr:          {desc_sim['radial_corr']:.4f}
  Quadrant corr:        {desc_sim['quadrant_corr']:.4f}
  Moran's I sim:        {desc_sim['morans_similarity']:.4f}
  Value hist corr:      {desc_sim['value_hist_corr']:.4f}

SPATIAL STATS:
  Gene Moran's I:       {gene_sig.morans_i:.4f}
  m/z Moran's I:        {mz_sig.morans_i:.4f}
        """
        axes[2, 3].text(0.05, 0.95, metrics_text, transform=axes[2, 3].transAxes,
                        fontsize=9, verticalalignment='top', family='monospace',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[2, 3].axis('off')
        plt.suptitle(f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name} | Score: {combined:.3f}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()

    def find_matches(self, gene_sig, mz_sigs, top_k=20):
        def compute_match(mz_name, mz_sig):
            coord_sim = compute_coordinate_based_similarity(gene_sig, mz_sig)
            desc_sim = compute_descriptor_similarity(gene_sig, mz_sig)
            combined = compute_combined_score(coord_sim, desc_sim)
            return {'gene': gene_sig.feature_name, 'rna_sample': gene_sig.sample_id,
                    'mz_feature': mz_name, 'msi_sample': mz_sig.sample_id,
                    **coord_sim, **desc_sim, 'combined_score': combined}
        matches = Parallel(n_jobs=self.n_jobs, prefer='threads')(
            delayed(compute_match)(mz_name, mz_sig) for mz_name, mz_sig in mz_sigs.items())
        return pd.DataFrame(matches).sort_values('combined_score', ascending=False).head(top_k)

    def run_analysis(self, top_k=20):
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V1 (Analytic - Optimized)")
        print(f"RNA: {RNA_PIXEL_SIZE}μm (hexagonal) | MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print("Strategy: Analytic Spatial Descriptors + Heuristic Importance")
        print("="*70)
        gene_avail = {gene: {s: gene in self.rna_data[s].var_names
                             for s in RNA_SAMPLE_IDS if s in self.rna_data}
                      for gene in AAD_TARGET_GENES}
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            n = sum(gene_avail[gene].values())
            print(f"  {gene}: {n}/{len(RNA_SAMPLE_IDS)}")
        all_results = []
        all_top5_results = []
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            available = [s for s, a in gene_avail[gene].items() if a]
            if not available:
                continue
            for rna_sample in available:
                print(f"\n  {rna_sample}")
                adata = self.rna_data[rna_sample]
                rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                gene_idx = list(adata.var_names).index(gene)
                gene_expr = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, 'toarray') else adata.X[:, gene_idx].flatten()
                msi_sample = rna_sample
                if msi_sample not in self.msi_data:
                    print(f"    MSI {msi_sample} not found")
                    continue
                msi_adata = self.msi_data[msi_sample]
                msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
                aligned_coords = align_rna_to_msi(rna_coords, msi_coords)
                rna_nn_indices = self._get_nn_indices(rna_coords, 6, f"rna_{rna_sample}")
                msi_nn_indices = self._get_nn_indices(msi_coords, 8, f"msi_{msi_sample}")
                gene_sig = self.extract_signature(rna_coords, gene_expr, rna_sample, gene, 'gene', 6, rna_nn_indices, aligned_coords)
                self.visualize_signature(gene_sig, os.path.join(self.output_dir, 'gene_visualizations', f'{gene}_{rna_sample}.png'))
                with open(os.path.join(self.output_dir, 'descriptors', f'{gene}_{rna_sample}_descriptors.pkl'), 'wb') as f:
                    pickle.dump({
                        'feature_name': gene_sig.feature_name, 'sample_id': gene_sig.sample_id,
                        'feature_type': gene_sig.feature_type, 'value_histogram': gene_sig.value_histogram,
                        'spatial_histogram': gene_sig.spatial_histogram, 'radial_profile': gene_sig.radial_profile,
                        'quadrant_stats': gene_sig.quadrant_stats, 'morans_i': gene_sig.morans_i,
                        'coordinates': gene_sig.coordinates, 'aligned_coordinates': gene_sig.aligned_coordinates,
                        'expression_stats': {
                            'mean': float(gene_sig.raw_values.mean()), 'std': float(gene_sig.raw_values.std()),
                            'min': float(gene_sig.raw_values.min()), 'max': float(gene_sig.raw_values.max()),
                            'median': float(np.median(gene_sig.raw_values)), 'n_spots': len(gene_sig.raw_values),
                            'n_nonzero': int((gene_sig.raw_values > 0).sum())},
                        'importance_stats': {
                            'mean': float(gene_sig.node_importance.mean()), 'std': float(gene_sig.node_importance.std()),
                            'top_10pct_threshold': float(np.percentile(gene_sig.node_importance, 90))}}, f)
                print(f"    Extracting MSI signatures...")
                msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
                mz_names = list(msi_adata.var_names)
                def extract_single_mz(i):
                    return mz_names[i], self.extract_signature(msi_coords, msi_data[:, i], msi_sample, mz_names[i], 'mz', 8, msi_nn_indices)
                mz_results = Parallel(n_jobs=self.n_jobs, prefer='threads')(delayed(extract_single_mz)(i) for i in range(len(mz_names)))
                mz_sigs = dict(mz_results)
                print(f"      {len(mz_names)} m/z features processed")
                print(f"    Matching...")
                matches = self.find_matches(gene_sig, mz_sigs, top_k)
                all_results.append(matches)
                top5_matches = matches.head(top_k).copy() # Store top k matches for detailed analysis
                top5_matches['gene'] = gene
                top5_matches['rna_sample'] = rna_sample
                top5_matches['rank'] = range(1, len(top5_matches) + 1)
                all_top5_results.append(top5_matches)
                if len(matches) > 0:
                    print(f"\n    Top {top_k}:")
                    for idx in range(min(top_k, len(matches))): # Show top k matches
                        m = matches.iloc[idx]
                        print(f"      {idx+1}. m/z {m['mz_feature']}: {m['combined_score']:.3f}")
                    top = matches.iloc[0]
                    top_mz = mz_sigs[top['mz_feature']]
                    coord_sim = compute_coordinate_based_similarity(gene_sig, top_mz)
                    desc_sim = compute_descriptor_similarity(gene_sig, top_mz)
                    self.visualize_match(gene_sig, top_mz, coord_sim, desc_sim,
                                         os.path.join(self.output_dir, 'match_visualizations', f'{gene}_{rna_sample}_top.png'))
                    for idx in range(min(top_k, len(matches))): # Save top k matches' descriptors
                        m = matches.iloc[idx]
                        mz_name = m['mz_feature']
                        mz_sig = mz_sigs[mz_name]
                        mz_filename = mz_name.replace('/', '_').replace('\\', '_')
                        with open(os.path.join(self.output_dir, 'descriptors',
                                               f'{gene}_{rna_sample}_match{idx+1}_{mz_filename}_descriptors.pkl'), 'wb') as f:
                            pickle.dump({
                                'gene': gene, 'gene_sample': rna_sample, 'mz_feature': mz_name,
                                'mz_sample': mz_sig.sample_id, 'match_rank': idx + 1,
                                'combined_score': float(m['combined_score']),
                                'mz_value_histogram': mz_sig.value_histogram, 'mz_spatial_histogram': mz_sig.spatial_histogram,
                                'mz_radial_profile': mz_sig.radial_profile, 'mz_quadrant_stats': mz_sig.quadrant_stats,
                                'mz_morans_i': mz_sig.morans_i, 'gene_value_histogram': gene_sig.value_histogram,
                                'gene_spatial_histogram': gene_sig.spatial_histogram, 'gene_radial_profile': gene_sig.radial_profile,
                                'gene_quadrant_stats': gene_sig.quadrant_stats, 'gene_morans_i': gene_sig.morans_i,
                                'similarity_scores': {k: float(v) if isinstance(v, (int, float, np.floating)) else v
                                                      for k, v in m.items() if k not in ['gene', 'rna_sample', 'mz_feature', 'msi_sample']}}, f)
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            results.to_csv(os.path.join(self.output_dir, 'gene_to_mz_matches_v1_analytic.csv'), index=False)
            if all_top5_results:
                top5_df = pd.concat(all_top5_results, ignore_index=True)
                priority_cols = ['gene', 'rna_sample', 'rank', 'mz_feature', 'msi_sample', 'combined_score']
                other_cols = [c for c in top5_df.columns if c not in priority_cols]
                top5_df = top5_df[priority_cols + other_cols]
                top5_df = top5_df.sort_values(['gene', 'rna_sample', 'rank'])
                top5_df.to_csv(os.path.join(self.output_dir, f'gene_to_mz_top{TOP_K_MATCHES}_matches_all_scores.csv'), index=False)
            print(f"\nSaved results to: {self.output_dir}")
            print("\n" + "="*70)
            print("TOP MATCHES")
            print("="*70)
            for gene in AAD_TARGET_GENES:
                gr = results[results['gene'] == gene]
                if len(gr) > 0:
                    t = gr.iloc[0]
                    print(f"\n{gene}: m/z {t['mz_feature']} ({t['rna_sample']}) = {t['combined_score']:.3f}")
            return results
        return None


def main():
    print("="*70)
    print("V1: Analytic Spatial Matching (No GNN/Transformer) - Optimized")
    print(f"RNA: {RNA_PIXEL_SIZE}μm | MSI: {MSI_PIXEL_SIZE}μm")
    print("="*70)
    matcher = AnalyticPatternMatcher(output_dir=f'./{TOP_K_MATCHES}_gene_to_mz_synced_results_v1_analytic_fast_new', n_jobs=-1)
    matcher.load_all_data()
    results = matcher.run_analysis(top_k=TOP_K_MATCHES)
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    return matcher, results


if __name__ == "__main__":
    matcher, results = main()

V1: Analytic Spatial Matching (No GNN/Transformer) - Optimized
RNA: 55μm | MSI: 60μm
Loading RNA-seq data...
  Pixel size: 55 μm (hexagonal)
  YC_1: (2112, 32285)
  YC_2: (2775, 32285)
  YC_3: (2808, 32285)
  YC_4: (2725, 32285)
  YAD_1: (2915, 32285)
  YAD_2: (2960, 32285)
  YAD_3: (2880, 32285)
  YAD_4: (2939, 32285)
  AC_1: (3065, 32285)
  AC_2: (3054, 32285)
  AC_3: (2892, 32285)
  AC_4: (3002, 32285)
  AAD_1: (2700, 32285)
  AAD_2: (2171, 32285)
  AAD_3: (1584, 32285)
  AAD_4: (2438, 32285)

Loading MSI data...
  Pixel size: 60 μm (Cartesian)
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

GENE-TO-M/Z MATCHING V1 (Analytic - Optimized)
RNA: 55μm (hexagonal) | MSI: 60μm (Cartesian)
Strategy: Analytic Spati

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.


    Top 26:
      1. m/z 769.5945: 0.449
      2. m/z 629.5422: 0.449
      3. m/z 840.641: 0.438
      4. m/z 518.3231: 0.423
      5. m/z 835.6707: 0.420
      6. m/z 826.6237: 0.417
      7. m/z 831.7023: 0.413
      8. m/z 994.6765: 0.410
      9. m/z 655.5653: 0.407
      10. m/z 990.6412: 0.405
      11. m/z 866.6629: 0.405
      12. m/z 768.5908: 0.404
      13. m/z 872.5589: 0.401
      14. m/z 993.6724: 0.400
      15. m/z 839.637: 0.399
      16. m/z 846.5419: 0.396
      17. m/z 772.5863: 0.395
      18. m/z 991.6472: 0.393
      19. m/z 755.5406: 0.392
      20. m/z 824.5555: 0.392
      21. m/z 845.5309: 0.389
      22. m/z 730.5358: 0.389
      23. m/z 758.5688: 0.388
      24. m/z 770.5648: 0.386
      25. m/z 759.5747: 0.384
      26. m/z 856.5819: 0.384

  YC_2
    Extracting MSI signatures...
      528 m/z features processed
    Matching...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(



    Top 26:
      1. m/z 770.6033: 0.503
      2. m/z 769.5945: 0.448
      3. m/z 864.6459: 0.442
      4. m/z 841.6461: 0.440
      5. m/z 755.5406: 0.438
      6. m/z 816.698: 0.434
      7. m/z 754.5364: 0.433
      8. m/z 835.6707: 0.432
      9. m/z 572.3314: 0.427
      10. m/z 759.6355: 0.427
      11. m/z 794.6002: 0.425
      12. m/z 862.6293: 0.422
      13. m/z 655.5653: 0.421
      14. m/z 768.5908: 0.420
      15. m/z 500.3132: 0.412
      16. m/z 400.3434: 0.409
      17. m/z 780.5501: 0.402
      18. m/z 817.699: 0.399
      19. m/z 387.2512: 0.393
      20. m/z 754.5892: 0.393
      21. m/z 994.6765: 0.393
      22. m/z 866.6629: 0.392
      23. m/z 725.5016: 0.392
      24. m/z 969.6712: 0.390
      25. m/z 857.5847: 0.390
      26. m/z 426.3589: 0.388

  YC_3
    Extracting MSI signatures...
      528 m/z features processed
    Matching...

    Top 26:
      1. m/z 599.5012: 0.615
      2. m/z 804.5508: 0.612
      3. m/z 698.4776: 0.611
      4. m/z 857.5847: 0.610

...


    Top 26:
      1. m/z 544.3043: 0.542
      2. m/z 545.3095: 0.508
      3. m/z 547.3226: 0.488
      4. m/z 458.3097: 0.460
      5. m/z 405.1784: 0.457
      6. m/z 745.5919: 0.451
      7. m/z 858.696: 0.439
      8. m/z 872.7073: 0.433
      9. m/z 859.6927: 0.421
      10. m/z 844.6774: 0.420
      11. m/z 630.6165: 0.418
      12. m/z 431.245: 0.417
      13. m/z 360.139: 0.416
      14. m/z 870.6936: 0.409
      15. m/z 530.3534: 0.408
      16. m/z 517.2901: 0.408
      17. m/z 487.2803: 0.400
      18. m/z 500.2857: 0.398
      19. m/z 933.7104: 0.394
      20. m/z 411.2503: 0.394
      21. m/z 351.1719: 0.392
      22. m/z 550.3825: 0.390
      23. m/z 397.1773: 0.389
      24. m/z 399.252: 0.384
      25. m/z 412.2818: 0.383
      26. m/z 785.652: 0.381

  YAD_4
    Extracting MSI signatures...
      528 m/z features processed
    Matching...

    Top 26:
      1. m/z 933.7104: 0.461
      2. m/z 693.4839: 0.406
      3. m/z 745.5919: 0.401
      4. m/z 550.3825: 0.391
 

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.


    Top 26:
      1. m/z 752.5583: 0.425
      2. m/z 770.5108: 0.420
      3. m/z 796.5214: 0.412
      4. m/z 846.5419: 0.386
      5. m/z 821.5308: 0.383
      6. m/z 544.3043: 0.373
      7. m/z 545.3095: 0.368
      8. m/z 527.3338: 0.366
      9. m/z 979.6568: 0.359
      10. m/z 748.6177: 0.359
      11. m/z 773.5294: 0.356
      12. m/z 793.5559: 0.356
      13. m/z 778.6205: 0.355
      14. m/z 518.3231: 0.348
      15. m/z 774.528: 0.347
      16. m/z 764.5203: 0.347
      17. m/z 749.6215: 0.347
      18. m/z 909.6691: 0.345
      19. m/z 881.6722: 0.342
      20. m/z 894.675: 0.342
      21. m/z 849.5611: 0.342
      22. m/z 653.5428: 0.341
      23. m/z 845.5309: 0.339
      24. m/z 858.593: 0.337
      25. m/z 740.4725: 0.336
      26. m/z 801.5554: 0.333

  YAD_1
    Extracting MSI signatures...
      528 m/z features processed
    Matching...

    Top 26:
      1. m/z 908.6916: 0.477
      2. m/z 907.6881: 0.452
      3. m/z 770.5108: 0.447
      4. m/z 915.6195: 0.445

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.


    Top 26:
      1. m/z 922.6372: 0.446
      2. m/z 907.6881: 0.390
      3. m/z 915.6195: 0.388
      4. m/z 629.5422: 0.378
      5. m/z 881.6722: 0.376
      6. m/z 471.2859: 0.370
      7. m/z 933.7104: 0.368
      8. m/z 722.4934: 0.368
      9. m/z 908.6916: 0.363
      10. m/z 548.539: 0.355
      11. m/z 730.5358: 0.354
      12. m/z 487.2803: 0.353
      13. m/z 517.2901: 0.352
      14. m/z 822.5954: 0.352
      15. m/z 580.3597: 0.352
      16. m/z 568.3391: 0.349
      17. m/z 840.5611: 0.348
      18. m/z 649.5163: 0.347
      19. m/z 773.6029: 0.345
      20. m/z 676.453: 0.345
      21. m/z 716.4498: 0.344
      22. m/z 740.4725: 0.344
      23. m/z 725.5016: 0.343
      24. m/z 551.352: 0.341
      25. m/z 565.3716: 0.340
      26. m/z 646.4416: 0.336

  AC_4
    Extracting MSI signatures...
      528 m/z features processed
    Matching...

    Top 26:
      1. m/z 568.3391: 0.508
      2. m/z 845.5309: 0.484
      3. m/z 778.6205: 0.483
      4. m/z 579.5304: 0.478
